# 00 · Colab setup

Run once per Colab session. Put the four unzipped v3 folders (NF-UNSW-NB15-v3, NF-ToN-IoT-v3, NF-BoT-IoT-v3, NF-CICIDS2018-v3) **or** the four zip files in `MyDrive/thesis_data/` first.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# The repo is private: clone with a read-only GitHub token stored in Colab Secrets
# (key icon on the left -> add secret named GH_TOKEN, turn on "Notebook access").
import os, subprocess
from google.colab import userdata
REPO = 'github.com/mirmumin/netflow-crossdataset-ids.git'
DEST = '/content/netflow-crossdataset-ids'
try:
    token = userdata.get('GH_TOKEN')
except Exception:
    token = None
url = f'https://{token}@{REPO}' if token else f'https://{REPO}'
if not os.path.exists(DEST):
    r = subprocess.run(['git', 'clone', url, DEST], capture_output=True, text=True)
    print((r.stdout + r.stderr).replace(token or '\0', '***'))
else:
    subprocess.run(['git', '-C', DEST, 'pull', '-q'])
assert os.path.exists(DEST), 'Clone failed: add the GH_TOKEN secret (see comment above) and run this cell again.'
%cd /content/netflow-crossdataset-ids
# Colab already has pandas, pyarrow, scikit-learn, pyyaml, scipy for this step.
# Installing the full pinned requirements.txt here would downgrade numpy and force a restart.
!pip install -q pyyaml pyarrow

In [ ]:
# Copy (or unzip) the data from Drive into data/raw/
import glob, os, shutil, zipfile
SRC = '/content/drive/MyDrive/thesis_data'
os.makedirs('data/raw', exist_ok=True)
found = glob.glob(f'{SRC}/*.zip') + [d for d in glob.glob(f'{SRC}/NF-*') if os.path.isdir(d)]
assert found, f'No zip files or NF-* folders in {SRC}. Upload the 4 zips to MyDrive/thesis_data first.'
for z in glob.glob(f'{SRC}/*.zip'):
    name = os.path.splitext(os.path.basename(z))[0]
    print('unzipping', z); zipfile.ZipFile(z).extractall(f'data/raw/{name}')
for d in glob.glob(f'{SRC}/NF-*'):
    if os.path.isdir(d):
        print('copying', d); shutil.copytree(d, f'data/raw/{os.path.basename(d)}', dirs_exist_ok=True)
!find data/raw -name '*.csv' -exec ls -lh {} \;

In [ ]:
!python scripts/prepare_data.py
!cat data/processed/schema_report.json
!head -40 data/processed/class_summary.csv

Copy the outputs of the last cell back to Claude. The processed parquet files are small; save them to Drive so later sessions skip this step:

In [ ]:
!mkdir -p /content/drive/MyDrive/thesis_data/processed && cp data/processed/* /content/drive/MyDrive/thesis_data/processed/